# Create and Load Molecule-Level Unmet Need Scoring Table in BigQuery

This notebook:
1. Authenticates to BigQuery with a service account JSON file
2. Executes the DDL script to create the `molecule_level_unmet_needs_scoring` table
3. Loads `unmet_needs_individual_scores.csv` data into BigQuery
4. Validates the loaded row count

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from pathlib import Path
import os
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

# Resolve unmet_needs root even when running from ddl_scripts/
NOTEBOOK_DIR = Path.cwd()
UNMET_NEEDS_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "ddl_scripts" else NOTEBOOK_DIR

SQL_FILE_PATH = UNMET_NEEDS_DIR / "ddl_scripts" / "sql_scripts" / "create_molecule_level_unmet_need_scoring_table.sql"

# Update this path if the CSV lives elsewhere
CSV_FILE_PATH = UNMET_NEEDS_DIR / "notebooks" / "unmet_needs_individual_scores.csv"

SERVICE_ACCOUNT_JSON_PATH = Path(
    os.getenv("GOOGLE_APPLICATION_CREDENTIALS", str(UNMET_NEEDS_DIR / "service_account.json"))
)

print(f"Unmet needs dir : {UNMET_NEEDS_DIR}")
print(f"SQL path exists : {SQL_FILE_PATH.exists()} -> {SQL_FILE_PATH}")
print(f"CSV path exists : {CSV_FILE_PATH.exists()} -> {CSV_FILE_PATH}")
print(f"Service JSON exists: {SERVICE_ACCOUNT_JSON_PATH.exists()} -> {SERVICE_ACCOUNT_JSON_PATH}")

Unmet needs dir : c:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\market_potential\unmet_needs
SQL path exists : True -> c:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\market_potential\unmet_needs\ddl_scripts\sql_scripts\create_molecule_level_unmet_need_scoring_table.sql
CSV path exists : True -> c:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\market_potential\unmet_needs\notebooks\unmet_needs_individual_scores.csv
Service JSON exists: True -> C:\Users\p00102977\Desktop\Projects\API\repo\cognito-prod-394707-d38a0283cb16.json


In [3]:
# Metadata sourced from ddl_scripts/molecule_level_unmet_need_scoring_schema.json
PROJECT_ID = "cognito-prod-394707"
DATASET_ID = "cognito_prod_datamart"
TABLE_NAME = "molecule_level_unmet_needs_scoring"

TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"

SCHEMA = [
    bigquery.SchemaField("unmet_need",                 "STRING"),
    bigquery.SchemaField("category",                    "STRING"),
    bigquery.SchemaField("country",                     "STRING"),
    bigquery.SchemaField("score_dimension",             "STRING"),
    bigquery.SchemaField("unmet_need_score",            "FLOAT64"),
    bigquery.SchemaField("evidence_impact_score",       "FLOAT64"),
    bigquery.SchemaField("unmet_need_score_rationale",  "STRING"),
    bigquery.SchemaField("evidence_score_rationale",    "STRING"),
    bigquery.SchemaField("confidence",                  "STRING"),
    bigquery.SchemaField("num_of_trials_assessed",      "INT64"),
    bigquery.SchemaField("evidence_impact_weight",      "FLOAT64"),
    bigquery.SchemaField("weighted_unmet_need_score",   "FLOAT64"),
]

print(f"Target table: {TABLE_ID}")

Target table: cognito-prod-394707.cognito_prod_datamart.molecule_level_unmet_needs_scoring


In [4]:
if not SERVICE_ACCOUNT_JSON_PATH.exists():
    raise FileNotFoundError(
        f"Service account JSON not found: {SERVICE_ACCOUNT_JSON_PATH}. "
        "Set GOOGLE_APPLICATION_CREDENTIALS or update SERVICE_ACCOUNT_JSON_PATH."
    )

credentials = service_account.Credentials.from_service_account_file(str(SERVICE_ACCOUNT_JSON_PATH))
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print(f"Authenticated BigQuery client for project: {client.project}")

Authenticated BigQuery client for project: cognito-prod-394707


In [5]:
if not SQL_FILE_PATH.exists():
    raise FileNotFoundError(f"SQL script not found: {SQL_FILE_PATH}")

ddl_sql = SQL_FILE_PATH.read_text(encoding="utf-8").strip()
if not ddl_sql:
    raise ValueError("SQL script is empty.")

ddl_job = client.query(ddl_sql)
ddl_job.result()
print(f"Table creation DDL executed successfully. Job ID: {ddl_job.job_id}")

Table creation DDL executed successfully. Job ID: 43273838-dcf4-4ee1-a733-05f5579c854f


In [ ]:
if not CSV_FILE_PATH.exists():
    raise FileNotFoundError(f"CSV file not found: {CSV_FILE_PATH}")

df = pd.read_csv(CSV_FILE_PATH)

required_cols = [field.name for field in SCHEMA]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"CSV missing required columns: {missing_cols}")

# Keep only table columns in schema order
df = df[required_cols].copy()

# Coerce STRING fields so pyarrow does not infer incompatible dtypes
string_cols = [field.name for field in SCHEMA if field.field_type == "STRING"]
for col in string_cols:
    df[col] = df[col].astype("string")
    df[col] = df[col].where(df[col].notna(), None)

# Coerce numeric fields
float_cols = [field.name for field in SCHEMA if field.field_type == "FLOAT64"]
int_cols   = [field.name for field in SCHEMA if field.field_type == "INT64"]
for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

print(f"CSV loaded: {len(df)} rows, {len(df.columns)} columns.")
df.head(3)

In [ ]:
job_config = bigquery.LoadJobConfig(
    schema=SCHEMA,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_dataframe(df, TABLE_ID, job_config=job_config)
load_job.result()
print(f"Data load completed. Job ID: {load_job.job_id}")
print(f"Rows written: {load_job.output_rows}")

In [ ]:
validation_query = f"""
SELECT
  COUNT(*)                       AS row_count,
  COUNT(DISTINCT unmet_need)     AS unique_unmet_needs,
  COUNT(DISTINCT country)        AS unique_countries,
  ROUND(AVG(unmet_need_score), 3) AS avg_unmet_need_score
FROM `{TABLE_ID}`
"""

result_df = client.query(validation_query).result().to_dataframe()
print("Validation summary:")
print(result_df.to_string(index=False))